[<< Sommaire QC](../README.md) | [Précédent : QC-Py-43-Diffusion-Series-Synthetiques <<](./QC-Py-43-Diffusion-Series-Synthetiques.ipynb)

# QC-Py-44 — Agent de trading multi-LLM : analystes, débat et mémoire

**Série QuantConnect / Python — volet P2 de l'EPIC #19306.**

Un LLM seul répond à une question. Une **firme de trading** fait autre chose : des analystes
travaillent des angles différents, un débat confronte les thèses, un gérant tranche, et surtout
**la firme se souvient** de ce que ses décisions passées ont produit. Le cadre
[TradingAgents](https://arxiv.org/abs/2412.20138) (Xiao, Sun, Luo & Wang, arXiv:2412.20138)
reproduit cette organisation avec des agents LLM ; ce carnet en reprend l'ossature à l'échelle
pédagogique, et mesure ce qu'elle apporte — ou n'apporte pas.

## Ce que ce carnet mesure

1. Le **cadre complet** : deux analystes, un débat bull/bear borné, un gérant (PM), et une
   **mémoire de décisions soldées** — chaque décision est journalisée, soldée au terme de
   l'horizon de détention contre le rendement réalisé, puis résumée en une phrase réinjectée
   dans les décisions suivantes.
2. Le **témoin négatif** : le même modèle, le même contexte numérique, **un seul appel**. Si le
   débat et la mémoire n'ajoutent rien, les deux bras doivent être indiscernables.
3. Le **verdict** de la doctrine §C du dépôt : quatre graines, pertes appariées,
   Diebold-Mariano, frais de transaction, et un verdict explicite — jamais « prometteur ».

## Ce que ce carnet n'est pas

Ce n'est pas une réplication de TradingAgents : le cadre original fait intervenir des
analystes fondamental, sentiment et technique sur des **actualités**, une équipe de risque et
un gestionnaire de portefeuille multi-actifs, et il est évalué sur des actions américaines.
Ici, deux analystes travaillent **les mêmes chiffres** sous deux lentilles différentes
(momentum et risque), sur **un seul actif** liquide, sans données textuelles. La conclusion
transférable est méthodologique : ce qui se mesure, c'est l'effet du protocole de délibération
et de la mémoire — pas la performance d'un système de production.

## 1. Le cadre, les rôles, et les organes natifs du dépôt

| Rôle | Ce qu'il reçoit | Ce qu'il rend |
|---|---|---|
| **Analyste momentum** | les indicateurs de la date de décision | biais, prévision chiffrée, argument |
| **Analyste risque** | les mêmes indicateurs, lentille réversion/volatilité | biais, prévision chiffrée, argument |
| **Bull** et **Bear** | le contexte **et** les deux analyses | un débat borné à deux tours |
| **PM** | contexte, analyses, débat, **mémoire soldée** | décision `LONG`/`FLAT`/`SHORT` + prévision |
| **Réflexion** | décision, prévision, réalisé | une leçon d'une phrase, stockée |

**Organes natifs (règle organ-first).** Le débat n'est pas réécrit ici : il passe par
`AgentGroupChat` de la série `GenAI/SemanticKernel/03-SemanticKernel-Agents.ipynb`, avec une
stratégie de sélection séquentielle et une borne de deux tours. Les rôles hors débat sont des
appels de complétion simples, comme dans `QC-Py-42`. Le **règlement** ne dépend d'aucune
opinion du modèle : il se fait sur les rendements réalisés du panier local, et le
Diebold-Mariano est le même code que celui de `QC-Py-42`.

**La question falsifiable.** Le cadre vaut-il mieux qu'un appel unique au même modèle sur le
même contexte ? Si le débat et la mémoire n'apportent rien, l'écart de perte entre les deux
bras doit être indistinguishable de zéro — et le verdict doit le dire.

In [1]:
import os, json, re, time, math, warnings
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)

APPELS = {"n": 0}            # compteur d'appels reels, imprime en fin de campagne

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY absente de l'environnement : le carnet exige des appels LLM reels "
        "(C.2). Aucune reponse simulee n'est committable. Exportez la cle puis relancez.")
print("OPENAI_API_KEY presente :", bool(os.environ.get("OPENAI_API_KEY")))

OPENAI_API_KEY presente : True


In [2]:
SEEDS = (0, 1, 7, 42)        # plancher multi-seed de la doctrine §C
HORIZON = 5                  # jours de detention, en seances
COST_BPS = 10                # frais par cote (crypto), en points de base
N_POINTS = 8                 # points de decision par graine
ESPACEMENT = 120             # jours calendaires entre deux points de decision
                             # (8 points couvrent ~2,3 ans : plusieurs regimes, pas un seul)
ACTIF = "BTC-USD"

MODELE = "gpt-5.6-luna"      # meme modele que QC-Py-26 et QC-Py-42 : la serie reste homogene
MAX_TOKENS = 260

print("modele   :", MODELE)
print("graines  :", SEEDS, "| horizon :", HORIZON, "seances | points :", N_POINTS)

modele   : gpt-5.6-luna
graines  : (0, 1, 7, 42) | horizon : 5 seances | points : 8


## 2. Les données

Le panier crypto local du dépôt (`datasets/yfinance/crypto_panier/`) sert de support : il est
versionné, donc le carnet ne dépend d'aucun accès réseau à l'exécution. On travaille sur un
seul actif — `BTC-USD` — parce que la décision porte sur une direction et une amplitude, et
que multiplier les actifs multiplierait le coût des appels sans changer la question posée.

In [3]:
def trouver_panier():
    ici = os.path.abspath(os.getcwd())
    for _ in range(7):
        cand = os.path.join(ici, "MyIA.AI.Notebooks", "QuantConnect",
                            "datasets", "yfinance", "crypto_panier")
        if os.path.isdir(cand):
            return cand
        ici = os.path.dirname(ici)
    raise FileNotFoundError("panier introuvable depuis %s" % os.getcwd())


PANIER = trouver_panier()
_rel = PANIER[PANIER.find("MyIA.AI.Notebooks"):] if "MyIA.AI.Notebooks" in PANIER else PANIER
fichier = os.path.join(PANIER, "%s_2018-01-01_2026-05-01.csv" % ACTIF)
brut = pd.read_csv(fichier, parse_dates=["Date"]).set_index("Date").sort_index()
prix = brut["Close"].astype(float).dropna()
rend = np.log(prix).diff().dropna()

print("panier     :", _rel)
print("actif      :", ACTIF)
print("periode    :", prix.index.min().date(), "->", prix.index.max().date())
print("seances    :", len(prix), "| rendements log :", len(rend))

panier     : MyIA.AI.Notebooks\QuantConnect\datasets\yfinance\crypto_panier
actif      : BTC-USD
periode    : 2018-01-01 -> 2026-04-30
seances    : 3042 | rendements log : 3041


In [4]:
def contexte(indice):
    # Tout ce que les agents voient : des nombres calcules, jamais une opinion.
    p = prix.iloc[indice]
    r20 = prix.iloc[indice] / prix.iloc[indice - 20] - 1.0
    v20 = rend.iloc[indice - 20:indice].std() * np.sqrt(365)
    haut60 = prix.iloc[indice - 60:indice + 1].max()
    dd = p / haut60 - 1.0
    ma20 = prix.iloc[indice - 20:indice].mean()
    z = (p - ma20) / prix.iloc[indice - 20:indice].std()
    sk = rend.iloc[indice - 60:indice].skew()
    cinq = rend.iloc[indice - 5:indice]
    return {
        "date": str(prix.index[indice].date()),
        "cloture": float(p),
        "ret20": float(r20),
        "vol20": float(v20),
        "drawdown60": float(dd),
        "zscore20": float(z),
        "skew60": float(sk),
        "cinq_jours": [round(float(x) * 100, 2) for x in cinq],
        "realise_5j": float(prix.iloc[indice + HORIZON] / p - 1.0),
    }


# Points de decision : espaces regulierement, du plus ancien au plus recent, en gardant
# la marge de HORIZON seances pour que chaque decision soit soldable.
dernier = len(prix) - 1 - HORIZON
INDICES = [dernier - ESPACEMENT * k for k in range(N_POINTS - 1, -1, -1)]
CTX = [contexte(i) for i in INDICES]

table = pd.DataFrame([{
    "date": c["date"],
    "ret20 %": round(c["ret20"] * 100, 1),
    "vol20 %": round(c["vol20"] * 100, 1),
    "dd60 %": round(c["drawdown60"] * 100, 1),
    "z20": round(c["zscore20"], 2),
    "realise 5j %": round(c["realise_5j"] * 100, 2),
} for c in CTX])
print("points de decision :", len(CTX))
print("dispersion des rendements realises a 5 j : ecart-type %.2f pts"
      % (table["realise 5j %"].std()))
table

points de decision : 8
dispersion des rendements realises a 5 j : ecart-type 4.70 pts


,date,ret20 %,vol20 %,dd60 %,z20,realise 5j %
0,2024-01-06,6.3,42.7,-2.2,0.87,5.41
1,2024-05-05,1.0,52.1,-12.4,0.33,-5.06
2,2024-09-02,-2.5,50.2,-13.4,-0.48,-8.41
3,2024-12-31,-7.7,43.2,-12.0,-1.29,5.23
4,2025-04-30,18.3,37.8,-0.8,1.15,0.57
5,2025-08-28,-3.6,38.7,-8.8,-1.04,-1.19
6,2025-12-26,-2.2,28.2,-23.5,-0.77,0.24
7,2026-04-25,12.5,40.6,-0.8,1.25,-1.68


## 3. Les rôles hors débat

Les analystes et le PM sont des appels de complétion simples. Deux précautions valent d'être
posées ici :

- **Le schéma de sortie est imposé.** Chaque rôle rend un JSON à clés fixes, extrait par une
  fonction tolérante, et une réponse non parseable est **comptée** puis remplacée par une
  décision neutre (`FLAT`, prévision 0) — jamais par une valeur inventée qui aurait l'air
  d'une réponse du modèle.
- **Aucun repli simulé.** Si l'API échoue après reprises, le carnet lève : une sortie simulée
  ne se commite pas (C.2).

In [5]:
from openai import OpenAI

client = OpenAI()          # lit OPENAI_API_KEY dans l'environnement


def appel(systeme, utilisateur, graine, max_tokens=MAX_TOKENS):
    # Un appel reel, avec reprises bornees. Toute defaillance finale leve.
    derniere = None
    for essai in range(4):
        try:
            rep = client.chat.completions.create(
                model=MODELE,
                messages=[{"role": "system", "content": systeme},
                          {"role": "user", "content": utilisateur}],
                max_completion_tokens=max_tokens,
                seed=graine * 1000 + essai,
            )
            APPELS["n"] += 1
            return rep.choices[0].message.content or ""
        except Exception as e:                      # noqa: BLE001 - on veut le message
            derniere = e
            time.sleep(2.0 * (essai + 1))
    raise RuntimeError("appel LLM defaillant apres 4 essais : %s" % derniere)


def extraire_json(txt):
    # Tolere les clôtures markdown et le bavardage autour de l'objet.
    m = re.search(r"\{.*\}", txt or "", re.S)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
    except Exception:
        return None
    return obj if isinstance(obj, dict) else None


def _nombre(x, defaut=0.0):
    try:
        v = float(x)
        return v if math.isfinite(v) else defaut
    except Exception:
        return defaut


def _texte_ctx(c):
    return ("Actif : %s | date : %s | cloture : %.2f\n"
            "Rendement cumule 20 seances : %+.2f%%\n"
            "Volatilite annualisee 20 seances : %.2f%%\n"
            "Drawdown vs plus haut 60 seances : %+.2f%%\n"
            "Z-score du prix vs moyenne 20 seances : %+.2f\n"
            "Asymetrie des 60 derniers rendements : %+.2f\n"
            "Rendements des 5 dernieres seances (%% ) : %s"
            % (ACTIF, c["date"], c["cloture"], c["ret20"] * 100, c["vol20"] * 100,
               c["drawdown60"] * 100, c["zscore20"], c["skew60"],
               ", ".join("%+.2f" % x for x in c["cinq_jours"])))


SCHEMA = ('Reponds STRICTEMENT par un objet JSON, sans texte autour : '
          '{"biais": "haussier"|"baissier"|"neutre", "prevision_pct": <nombre>, '
          '"argument": "<deux phrases maximum, avec les chiffres du contexte>"}. '
          '"prevision_pct" est le rendement attendu en pourcentage sur les 5 prochaines seances.')

SYST_MOMENTUM = ("Tu es analyste momentum sur un desk crypto. Tu juges la poursuite ou "
                 "l'epuisement d'une tendance a partir des rendements, du drawdown et de la "
                 "position par rapport a la moyenne. " + SCHEMA)
SYST_RISQUE = ("Tu es analyste risque sur un desk crypto. Tu juges la compression ou "
               "l'expansion de la volatilite, la asymetrie des rendements et le retour a la "
               "moyenne apres un mouvement extreme. " + SCHEMA)
SYST_PM = ('Tu es le gestionnaire de portefeuille. Tu tranches apres avoir lu deux analyses '
           'et un debat. Reponds STRICTEMENT par un objet JSON : '
           '{"decision": "LONG"|"FLAT"|"SHORT", "prevision_pct": <nombre>, '
           '"confiance": <nombre entre 0 et 1>, "justification": "<deux phrases>"}.')


@dataclass
class Reponse:
    biais: str = "neutre"          # pour les analystes
    decision: str = "FLAT"         # pour le PM et la baseline
    prevision: float = 0.0
    texte: str = ""
    brut: str = ""
    parse: bool = True


def analyste(systeme, c, graine):
    brut = appel(systeme, _texte_ctx(c), graine)
    obj = extraire_json(brut)
    if obj is None or "prevision_pct" not in obj:
        return Reponse(texte=brut[:200], brut=brut, parse=False)
    return Reponse(biais=str(obj.get("biais", "neutre")).lower(),
                   prevision=_nombre(obj.get("prevision_pct")),
                   texte=str(obj.get("argument", ""))[:400], brut=brut)


print("client pret | appels :", APPELS["n"])

client pret | appels : 0


## 4. Le débat bull/bear — organe `AgentGroupChat`

Le débat est le seul endroit où les agents se parlent. Il est borné à **deux tours**
(Bull, Bear, Bull, Bear) : au-delà, les positions se répètent et le coût croît sans
information. Chaque intervenant reçoit le contexte chiffré **et** les deux analyses, et doit
chiffrer sa position — un débat sans nombre ne se solde pas.

In [6]:
from semantic_kernel.agents import AgentGroupChat, ChatCompletionAgent
from semantic_kernel.agents.strategies import SequentialSelectionStrategy
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import AuthorRole, ChatMessageContent

SERVICE = OpenAIChatCompletion(ai_model_id=MODELE)   # lit OPENAI_API_KEY


async def debattre(c, a_momentum, a_risque, tours=2):
    # Deux agents, selection sequentielle, arret dur apres 2*tours messages.
    consigne = _texte_ctx(c) + (
        "\n\nAnalyse momentum : biais %s, prevision %+.2f%%, argument : %s"
        "\nAnalyse risque   : biais %s, prevision %+.2f%%, argument : %s"
        % (a_momentum.biais, a_momentum.prevision, a_momentum.texte,
           a_risque.biais, a_risque.prevision, a_risque.texte))

    bull = ChatCompletionAgent(
        service=SERVICE, name="Bull",
        instructions=("Tu defends le scenario haussier. Deux phrases maximum. Tu dois citer "
                      "un chiffre du contexte et avancer une prevision chiffree a 5 seances."))
    bear = ChatCompletionAgent(
        service=SERVICE, name="Bear",
        instructions=("Tu defends le scenario baissier. Deux phrases maximum. Tu dois citer "
                      "un chiffre du contexte et avancer une prevision chiffree a 5 seances."))

    chat = AgentGroupChat(
        agents=[bull, bear],
        selection_strategy=SequentialSelectionStrategy(initial_agent_name="Bull"),
    )
    await chat.add_chat_message(ChatMessageContent(role=AuthorRole.USER, content=consigne))

    messages = []
    async for m in chat.invoke():
        messages.append((m.name or "?", (m.content or "").strip()))
        APPELS["n"] += 1
        if len(messages) >= 2 * tours:
            break
    return messages


# Demonstration sur le premier point de decision, avant la campagne complete.
demo = await debattre(CTX[0], analyste(SYST_MOMENTUM, CTX[0], 0), analyste(SYST_RISQUE, CTX[0], 0))
for nom, texte in demo:
    print("[%s] %s" % (nom, texte.replace("\n", " ")[:230]))
print("\nmessages de debat :", len(demo), "| appels cumules :", APPELS["n"])

[Bull] Le rendement cumulé de **+6,34 % sur 20 séances** confirme un momentum haussier, malgré une volatilité élevée.   Sur les **5 prochaines séances**, le BTC-USD pourrait progresser d’environ **+1,80 %**, vers **44 781 $**.
[Bear] Malgré le gain cumulé de **+6,34 % sur 20 séances**, la volatilité annualisée élevée de **42,70 %** et les cinq dernières séances légèrement négatives plaident pour une correction.   Sur les **5 prochaines séances**, le BTC-USD po
[Bull] Le rendement cumulé de **+6,34 % sur 20 séances** confirme un momentum haussier, malgré une volatilité élevée.   Sur les **5 prochaines séances**, le BTC-USD pourrait gagner environ **+1,80 %**, atteignant **44 781 $**.
[Bear] La volatilité annualisée élevée de **42,70 %** et les cinq dernières séances légèrement négatives signalent un risque de correction malgré le gain de **+6,34 %** sur 20 séances.   Sur les cinq prochaines séances, le BTC-USD pourra

messages de debat : 4 | appels cumules : 6


## 5. La mémoire de décisions soldées

C'est la pièce que le cadre ajoute à un débat : une décision n'est pas seulement prise, elle
est **soldée**. À l'échéance de détention, l'écart entre la prévision et le rendement réalisé
devient une leçon d'une phrase, et cette leçon revient dans le prompt du gérant.

Deux propriétés rendent la mémoire mesurable :

- **Au premier point de décision, elle est vide** — c'est le témoin négatif structurel : si le
  cadre gagne, ce n'est pas grâce à elle, puisqu'elle ne peut pas encore exister.
- **Chaque entrée est soldée avant d'être lue.** Les points sont espacés de 30 jours et
  l'horizon est de 5 séances : aucune entrée non soldée ne peut entrer dans le prompt. Le
  carnet vérifie cette propriété et lève si elle est violée.

In [7]:
class Memoire:
    # Journal de decisions soldees. Rien n'entre ici tant que le rendement est inconnu.
    def __init__(self):
        self.entrees = []

    def ajouter(self, date, decision, prevision, realise, reflexion):
        self.entrees.append({"date": date, "decision": decision,
                             "prevision": prevision, "realise": realise,
                             "reflexion": reflexion})

    def toutes_soldees(self):
        return all(e["realise"] is not None for e in self.entrees)

    def resume(self, k=3):
        soldees = [e for e in self.entrees if e["realise"] is not None]
        if not soldees:
            return "(memoire vide : aucune decision soldee a ce stade)"
        lignes = []
        for e in soldees[-k:]:
            lignes.append("- %s : %s, prevu %+.2f%%, realise %+.2f%% -> %s"
                          % (e["date"], e["decision"], e["prevision"],
                             e["realise"] * 100, e["reflexion"]))
        return "\n".join(lignes)

    def __len__(self):
        return len(self.entrees)


def pm(c, a_momentum, a_risque, debat, memoire, graine):
    fil = "\n".join("[%s] %s" % (n, t) for n, t in debat)
    demand = (_texte_ctx(c)
              + "\n\nAnalyse momentum : %s (%+.2f%%)\nAnalyse risque : %s (%+.2f%%)"
              % (a_momentum.biais, a_momentum.prevision, a_risque.biais, a_risque.prevision)
              + "\n\nDebat :\n" + fil
              + "\n\nMemoire des decisions soldees (les plus recentes) :\n" + memoire.resume(3)
              + "\n\nTranche. Rappelle-toi que la memoire ne contient que des decisions "
                "deja soldees contre le rendement realise.")
    brut = appel(SYST_PM, demand, graine)
    obj = extraire_json(brut)
    if obj is None or "decision" not in obj:
        return Reponse(brut=brut, parse=False)
    dec = str(obj.get("decision", "FLAT")).upper()
    dec = dec if dec in ("LONG", "FLAT", "SHORT") else "FLAT"
    return Reponse(decision=dec, prevision=_nombre(obj.get("prevision_pct")),
                   texte=str(obj.get("justification", ""))[:400], brut=brut)


SYST_REFLEXION = ("Tu tires une lecon d'une decision soldee. Reponds par UNE phrase, "
                  "reutilisable pour la prochaine decision, sans compliment ni excuse.")


def reflexion(decision, prevision, realise, graine):
    q = ("Decision : %s. Prevision : %+.2f%%. Rendement realise sur la periode de detention : "
         "%+.2f%%. Quelle lecon d'une phrase ?" % (decision, prevision, realise * 100))
    return appel(SYST_REFLEXION, q, graine, max_tokens=90).strip().replace("\n", " ")[:240]


memoire_test = Memoire()
print("memoire vide ->", len(memoire_test), "entree(s)")
print(memoire_test.resume())

memoire vide -> 0 entree(s)
(memoire vide : aucune decision soldee a ce stade)


## 6. Le bras baseline

Le témoin est le même modèle, le même contexte chiffré, **un seul appel** : ni analyste, ni
débat, ni mémoire. C'est le concurrent honnête — si le cadre multi-agents ne le bat pas, c'est
que le protocole n'apporte rien que le modèle ne savait déjà faire seul.

In [8]:
SYST_BASELINE = ('Tu es analyste de marche. A partir du contexte, prevois le rendement des 5 '
                 'prochaines seances. Reponds STRICTEMENT par un objet JSON : '
                 '{"prevision_pct": <nombre>, "decision": "LONG"|"FLAT"|"SHORT", '
                 '"argument": "<une phrase>"}.')


def baseline(c, graine):
    brut = appel(SYST_BASELINE, _texte_ctx(c), graine)
    obj = extraire_json(brut)
    if obj is None or "prevision_pct" not in obj:
        return Reponse(brut=brut, parse=False)
    dec = str(obj.get("decision", "FLAT")).upper()
    dec = dec if dec in ("LONG", "FLAT", "SHORT") else "FLAT"
    return Reponse(decision=dec, prevision=_nombre(obj.get("prevision_pct")),
                   texte=str(obj.get("argument", ""))[:300], brut=brut)


avant = APPELS["n"]
essai_bl = baseline(CTX[0], 0)
print("baseline sur %s : decision %s, prevision %+.2f%% (%s)"
      % (CTX[0]["date"], essai_bl.biais, essai_bl.prevision,
         "parse" if essai_bl.parse else "NON PARSE"))
print("appels consommes par le bras baseline :", APPELS["n"] - avant)

baseline sur 2024-01-06 : decision neutre, prevision +0.80% (parse)
appels consommes par le bras baseline : 1


## 7. La campagne

Chaque graine parcourt les huit points de décision **dans l'ordre chronologique** : c'est la
seule façon dont la mémoire peut se remplir et être lue. Pour chaque point, les deux bras
tournent sur le **même contexte**, ce qui rend les pertes appariées.

Coût : à chaque point et par graine, 2 analyses + 4 messages de débat + 1 décision de PM +
1 réflexion + 1 baseline = **9 appels**. Soit `8 x 4 x 9 = 288` appels — la cellule suivante
est de loin la plus longue du carnet.

In [9]:
def perte(prevision_pct, realise):
    # Perte de prevision : erreur absolue en points de pourcentage, comparable entre bras.
    return abs(prevision_pct / 100.0 - realise)


lignes = []
debut = time.time()

for graine in SEEDS:
    memoire = Memoire()
    for i, c in enumerate(CTX):
        a_mom = analyste(SYST_MOMENTUM, c, graine)
        a_ris = analyste(SYST_RISQUE, c, graine)
        debat = await debattre(c, a_mom, a_ris)
        decision = pm(c, a_mom, a_ris, debat, memoire, graine)

        # La memoire lue par le PM ne doit contenir que des decisions soldees.
        assert memoire.toutes_soldees(), "memoire non soldee lue par le PM"

        base = baseline(c, graine)
        realise = c["realise_5j"]

        lecon = reflexion(decision.decision, decision.prevision, realise, graine)
        memoire.ajouter(c["date"], decision.decision, decision.prevision, realise, lecon)

        lignes.append({
            "graine": graine, "date": c["date"], "realise": realise,
            "prev_pm": decision.prevision, "dec_pm": decision.decision,
            "prev_bl": base.prevision, "dec_bl": base.decision,
            "parse_pm": decision.parse, "parse_bl": base.parse,
            "biais_mom": a_mom.biais, "biais_ris": a_ris.biais,
            "memoire_avant": i, "parse_analystes": int(a_mom.parse) + int(a_ris.parse),
        })
    print("graine %-3d terminee | appels cumules %d" % (graine, APPELS["n"]))

res = pd.DataFrame(lignes)
res["perte_pm"] = [perte(p, r) for p, r in zip(res["prev_pm"], res["realise"])]
res["perte_bl"] = [perte(p, r) for p, r in zip(res["prev_bl"], res["realise"])]
res["d"] = res["perte_pm"] - res["perte_bl"]      # negatif = le cadre fait mieux

print("\nduree      : %.1f min | appels LLM : %d" % ((time.time() - debut) / 60.0, APPELS["n"]))
print("paires     :", len(res))
print("non parsees: PM %d, baseline %d, analystes %d/%d"
      % ((~res["parse_pm"]).sum(), (~res["parse_bl"]).sum(),
         len(res) * 2 - res["parse_analystes"].sum(), len(res) * 2))
res.head(8)

graine 0   terminee | appels cumules 79


graine 1   terminee | appels cumules 151


graine 7   terminee | appels cumules 223


graine 42  terminee | appels cumules 295

duree      : 12.3 min | appels LLM : 295
paires     : 32
non parsees: PM 0, baseline 0, analystes 14/64


,graine,date,realise,prev_pm,dec_pm,prev_bl,dec_bl,parse_pm,parse_bl,biais_mom,biais_ris,memoire_avant,parse_analystes,perte_pm,perte_bl,d
0,0,2024-01-06,0.054090,1.2,LONG,1.2,LONG,True,True,haussier,neutre,0,1,0.042090,0.042090,0.000000
1,0,2024-05-05,-0.050575,-0.5,FLAT,1.8,LONG,True,True,neutre,neutre,1,1,0.045575,0.068575,-0.023000
2,0,2024-09-02,-0.084124,-0.5,FLAT,-0.8,FLAT,True,True,neutre,neutre,2,0,0.079124,0.076124,0.003000
3,0,2024-12-31,0.052294,-0.5,FLAT,-0.8,SHORT,True,True,baissier,haussier,3,2,0.057294,0.060294,-0.003000
4,0,2025-04-30,0.005740,2.0,LONG,1.2,LONG,True,True,haussier,neutre,4,2,0.014260,0.006260,0.008000
5,0,2025-08-28,-0.011944,-1.2,SHORT,-0.6,FLAT,True,True,baissier,neutre,5,1,0.000056,0.005944,-0.005888
6,0,2025-12-26,0.002376,0.0,FLAT,-1.1,SHORT,True,True,baissier,haussier,6,2,0.002376,0.013376,-0.011000
7,0,2026-04-25,-0.016849,1.8,LONG,1.2,LONG,True,True,haussier,haussier,7,2,0.034849,0.028849,0.006000


## 8. Le verdict — doctrine §C

Trois conditions, **toutes** requises pour `BEATS` :

1. l'écart de perte `edge = moyenne(perte_baseline) - moyenne(perte_cadre)` est **au moins
   deux écarts-types** de sa dispersion entre graines ;
2. le Diebold-Mariano sur les pertes appariées est significatif (`p < 0,05`) ;
3. sa statistique est **négative** — c'est-à-dire que c'est bien le cadre qui a la perte la
   plus faible.

À quoi s'ajoute la lecture économique : la position implicite (`LONG`/`SHORT`/`FLAT`) est
tenue sur l'horizon, nette de **10 points de base par côté**.

In [10]:
def diebold_mariano(d, h=1):
    # Newey-West HAC, lag h. d = perte_modele - perte_baseline ; stat < 0 => modele meilleur.
    d = np.asarray(d, dtype=float)
    n = len(d)
    m = d.mean()
    gamma0 = np.sum((d - m) ** 2) / n
    s = gamma0
    for lag in range(1, h + 1):
        g = np.sum((d[lag:] - m) * (d[:-lag] - m)) / n
        s += 2.0 * (1.0 - lag / (h + 1.0)) * g
    stat = m / math.sqrt(s / n) if s > 0 else 0.0
    p = math.erfc(abs(stat) / math.sqrt(2.0))
    return float(m), float(stat), float(p)


par_graine = res.groupby("graine").apply(
    lambda g: pd.Series({"perte_pm": g["perte_pm"].mean(),
                         "perte_bl": g["perte_bl"].mean()}))
par_graine["edge"] = par_graine["perte_bl"] - par_graine["perte_pm"]

edge = float(par_graine["edge"].mean())
sigma = float(par_graine["edge"].std(ddof=1))
ratio = edge / sigma if sigma > 0 else float("inf")
moy_d, stat, pval = diebold_mariano(res["d"])

print(par_graine.round(5).to_string())
print("\nedge (points de %%)      : %+.4f" % (edge * 100))
print("sigma inter-graines     : %.4f  (n=%d)" % (sigma * 100, len(SEEDS)))
print("edge / sigma            : %+.2f  (seuil §C : >= 2)" % ratio)
print("Diebold-Mariano         : stat %+.3f, p = %.4f (seuil : p < 0,05 et stat < 0)"
      % (stat, pval))

conjonction = (ratio >= 2.0) and (pval < 0.05) and (stat < 0)
if conjonction:
    verdict = "BEATS"
elif ratio <= -2.0 and pval < 0.05 and stat > 0:
    verdict = "NO BEATS"
else:
    verdict = "INCONCLUSIVE"
print("\nVERDICT :", verdict)

        perte_pm  perte_bl     edge
graine                             
0        0.03445   0.03769  0.00324
1        0.03695   0.03994  0.00299
7        0.03969   0.03594 -0.00375
42       0.03569   0.03983  0.00414

edge (points de %)      : +0.1653
sigma inter-graines     : 0.3636  (n=4)
edge / sigma            : +0.45  (seuil §C : >= 2)
Diebold-Mariano         : stat -0.959, p = 0.3375 (seuil : p < 0,05 et stat < 0)

VERDICT : INCONCLUSIVE


In [11]:
# Lecture economique : la position implicite de chaque bras, nette de frais.
def pnl(decisions, realises, cout_bps=COST_BPS):
    signe = {"LONG": 1.0, "SHORT": -1.0, "FLAT": 0.0}
    p = []
    for dec, r in zip(decisions, realises):
        s = signe.get(dec, 0.0)
        frais = (2.0 * cout_bps / 10000.0) if s != 0.0 else 0.0
        p.append(s * r - frais)
    return np.asarray(p, dtype=float)


p_pm = pnl(res["dec_pm"], res["realise"])
p_bl = pnl(res["dec_bl"], res["realise"])
p_bh = pnl(["LONG"] * len(res), res["realise"])       # achat-conservation, meme cout

def sharpe_par_point(p):
    # Rendements par decision (et non journaliers) : on annualise par le nombre de
    # decisions par an, soit ~365/ESPACEMENT.
    m, s = p.mean(), p.std(ddof=1)
    if s == 0:
        return 0.0
    return float(m / s * np.sqrt(365.0 / ESPACEMENT))


print("Sharpe par decision (net de frais) :")
print("  cadre multi-agents : %+.3f" % sharpe_par_point(p_pm))
print("  baseline           : %+.3f" % sharpe_par_point(p_bl))
print("  achat-conservation : %+.3f" % sharpe_par_point(p_bh))
print("\nrendement moyen par decision : cadre %+.3f%% | baseline %+.3f%% | achat %+.3f%%"
      % (p_pm.mean() * 100, p_bl.mean() * 100, p_bh.mean() * 100))
print("frais retenus : %d bps par cote, aller-retour = %d bps" % (COST_BPS, 2 * COST_BPS))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(["cadre", "baseline", "achat"], [sharpe_par_point(p_pm), sharpe_par_point(p_bl),
                                        sharpe_par_point(p_bh)],
       color=["#2b6cb0", "#c05621", "#718096"])
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Sharpe par decision")
ax.set_title("QC-Py-44 - rendement risque net de frais, par bras")
plt.tight_layout()
plt.show()

Sharpe par decision (net de frais) :
  cadre multi-agents : +0.168
  baseline           : -0.682
  achat-conservation : -0.317

rendement moyen par decision : cadre +0.289% | baseline -1.308% | achat -0.812%
frais retenus : 10 bps par cote, aller-retour = 20 bps


In [12]:
# Temoins et rapport de biais : ce que le cadre a reellement utilise.
# Tous les chiffres ci-dessous se calculent sur `res` : aucun appel LLM supplementaire.
accord = (res["biais_mom"] == res["biais_ris"]).mean()

print("=== Temoin : la memoire etait-elle vide au premier point ? ===")
print("  entrees en memoire au point 1, par construction : 0")
print("  entrees atteintes en fin de graine              :", N_POINTS - 1)
print("  colonne `memoire_avant` de la campagne          :", [int(x) for x in sorted(res["memoire_avant"].unique())])

print("\n=== Les deux analystes se contredisent-ils ? ===")
print("  taux d'accord momentum/risque : %.1f%% (%d/%d)"
      % (accord * 100, int(accord * len(res)), len(res)))

print("\n=== Biais signe moyen (prevision - realise), par bras ===")
for nom, col in (("cadre", "prev_pm"), ("baseline", "prev_bl")):
    b = (res[col] / 100.0 - res["realise"]).mean()
    print("  %-9s : %+.4f pt  (%s)" % (nom, b * 100,
                                        "surestime" if b > 0 else "sous-estime"))

print("\n=== Ecart-type des previsions (amplitude osee par chaque bras) ===")
print("  cadre    : %.3f pt" % (res["prev_pm"] / 100.0).std())
print("  baseline : %.3f pt" % (res["prev_bl"] / 100.0).std())

print("\n=== Distribution des decisions ===")
print(pd.DataFrame({"cadre": res["dec_pm"].value_counts(),
                    "baseline": res["dec_bl"].value_counts()}).fillna(0).astype(int))

=== Temoin : la memoire etait-elle vide au premier point ? ===
  entrees en memoire au point 1, par construction : 0
  entrees atteintes en fin de graine              : 7
  colonne `memoire_avant` de la campagne          : [0, 1, 2, 3, 4, 5, 6, 7]

=== Les deux analystes se contredisent-ils ? ===
  taux d'accord momentum/risque : 40.6% (13/32)

=== Biais signe moyen (prevision - realise), par bras ===
  cadre     : +0.7999 pt  (surestime)
  baseline  : +0.7468 pt  (surestime)

=== Ecart-type des previsions (amplitude osee par chaque bras) ===
  cadre    : 0.012 pt
  baseline : 0.011 pt

=== Distribution des decisions ===
       cadre  baseline
LONG      12        14
FLAT      12         9
SHORT      8         9


## 9. Lecture du verdict

Trois choses se lisent ici, et il faut les séparer.

**Le verdict §C d'abord.** Il porte sur les **pertes de prévision** appariées, pas sur le
Sharpe : c'est la seule grandeur pour laquelle le carnet ait un critère de significativité
posé d'avance. Le Sharpe par décision est imprimé à titre descriptif — avec huit points de
décision et quatre graines, il n'a pas la puissance nécessaire pour trancher, et le carnet ne
s'en sert pas pour conclure.

**Le coût ensuite.** 288 appels pour comparer deux protocoles sur huit dates : c'est le prix
réel du cadre, et il est hors de portée d'un usage quotidien sur un univers large. Un cadre
multi-agents se justifie par un gain **mesurable**, jamais par son élégance.

**La mémoire enfin.** Le carnet ne l'isole pas : il n'y a pas de troisième bras « cadre sans
mémoire ». Ce qui est montré, c'est qu'elle est **structurellement vide au premier point** et
qu'elle ne contient que des décisions soldées — deux propriétés nécessaires, pas suffisantes.
Isoler son effet demande un bras supplémentaire, laissé en exercice.

## 10. Exercices

**Exercice 1 — un troisième analyste.** Ajoutez un analyste de flux qui ne regarde que
`cinq_jours` (les cinq derniers rendements) et mesurez si le débat change de camp plus
souvent. Le coût augmente de 8 appels par graine.

**Exercice 2 — la mémoire par similarité.** Remplacez `Memoire.resume` (les trois dernières
entrées) par une mémoire qui rend les trois entrées dont le contexte est le plus **proche** de
la décision courante (distance euclidienne sur `ret20`, `vol20`, `zscore20`, normalisés). La
question devient : la mémoire aide-t-elle parce qu'elle est récente, ou parce qu'elle est
pertinente ?

**Exercice 3 — l'horizon.** Faites varier `HORIZON` (1, 5, 20 séances). Un cadre qui gagne à
5 séances gagne-t-il encore à 20 ? Le verdict porte-t-il sur le protocole ou sur une
coïncidence d'horizon ?

In [13]:
# --- Exercice 1 : un troisieme analyste (flux) --------------------------------
SYST_FLUX = ("Tu es analyste de flux. Tu ne regardes que les cinq derniers rendements et tu "
             "juges la persistance court terme. " + SCHEMA)

def analyste_flux(c, graine):
    # TODO etudiant : appeler `analyste(SYST_FLUX, c, graine)` et renvoyer la Reponse.
    print("Exercice a completer : analyste de flux")
    return None

print("Exercice 1 pret a completer.")

Exercice 1 pret a completer.


In [14]:
# --- Exercice 2 : memoire par similarite de contexte -------------------------

def resume_par_similarite(memoire, c, k=3):
    # TODO etudiant : normaliser ret20, vol20, zscore20 sur les entrees soldees,
    # calculer la distance au contexte courant `c`, rendre les k plus proches.
    print("Exercice a completer : memoire par similarite")
    return "(exercice non complete)"

print("Exercice 2 pret a completer.")

Exercice 2 pret a completer.


In [15]:
# --- Exercice 3 : l'horizon de detention -------------------------------------

def campage_horizon(h):
    # TODO etudiant : reconstruire INDICES et CTX avec cet horizon, relancer la campagne,
    # imprimer le verdict §C. Attention : la marge de soldage depend de HORIZON.
    print("Exercice a completer : campagne a horizon %d" % h)

print("Exercice 3 pret a completer.")

Exercice 3 pret a completer.


## 11. Ce que ce carnet ne fait pas

- **Aucune donnée textuelle.** Les analystes de TradingAgents lisent des actualités, des
  états financiers et des réseaux sociaux. Ici, tout est numérique : le carnet mesure un
  protocole de décision, pas une capacité de lecture.
- **Un seul actif, un seul marché.** Les conclusions ne se transportent pas à un portefeuille
  multi-actifs ni à des actions.
- **Pas d'équipe de risque ni de contrainte d'exposition.** Le PM décide d'une direction et
  d'une amplitude, pas d'une taille de position dans un portefeuille.
- **La mémoire n'est pas isolée.** Voir la section 9 : un troisième bras serait nécessaire.
- **Huit points de décision.** C'est peu. Le Diebold-Mariano porte sur 32 paires, ce qui suffit
  à détecter un écart franc, pas un écart fin ; un verdict `INCONCLUSIVE` doit se lire comme
  « pas de preuve », jamais comme « pas d'effet ».

## Références

- Xiao, Sun, Luo & Wang — *TradingAgents: Multi-Agents LLM Financial Trading Framework*,
  arXiv:2412.20138 (v7, 2025-06-03). PDF archivé au gisement bibliographique du dépôt.
- `GenAI/SemanticKernel/03-SemanticKernel-Agents.ipynb` — organe `AgentGroupChat`.
- `QC-Py-42-Alpha-Mining-Evolution.ipynb` — doctrine §C, Diebold-Mariano local, coûts.

[<< Sommaire QC](../README.md) | [Précédent : QC-Py-43-Diffusion-Series-Synthetiques <<](./QC-Py-43-Diffusion-Series-Synthetiques.ipynb)